# LLM Paper Extraction

Interactive notebook for extracting structured information from papers using local LLM (Ollama).

**Features:**
- Comprehensive extraction with discussion, research context, and future directions
- Text quality assessment to identify problematic papers
- Vision-based extraction for ILL-contaminated PDFs
- Batch extraction with progress tracking

## Cell 1: Setup & Imports

In [ ]:
import sys
import json
import asyncio
from pathlib import Path
from datetime import datetime

# Add src to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Imports
import httpx
from tqdm.notebook import tqdm
from IPython.display import display, Markdown, HTML

from config.ai_settings import settings
from services.paper_service import PaperService
from services.extraction_service import ExtractionService, PaperExtraction
from literature_core import get_session, Paper, PaperContent

print(f"Project root: {PROJECT_ROOT}")
print(f"Python path configured")

## Cell 2: Check Ollama Status

In [ ]:
def check_ollama():
    """Check Ollama status and list models."""
    print("=" * 60)
    print("OLLAMA STATUS")
    print("=" * 60)
    
    try:
        with httpx.Client(timeout=5) as client:
            response = client.get(f"{settings.ollama.host}/api/tags")
            if response.status_code == 200:
                print(f"✓ Ollama is running at {settings.ollama.host}")
                data = response.json()
                print(f"\nInstalled models:")
                for m in data.get('models', []):
                    size_gb = m['size'] / (1024**3)
                    current = " ← CURRENT" if m['name'] == settings.ollama.reader_model else ""
                    print(f"  • {m['name']}: {size_gb:.1f} GB{current}")
            else:
                print(f"✗ Ollama returned status {response.status_code}")
    except Exception as e:
        print(f"✗ Cannot connect to Ollama: {e}")
        print(f"  Make sure Ollama is running: ollama serve")
    
    print(f"\nConfiguration:")
    print(f"  Reader model: {settings.ollama.reader_model}")
    print(f"  Temperature: {settings.ollama.reader_temperature}")
    print(f"  Timeout: {settings.ollama.timeout}s")

check_ollama()

## Cell 3: Select Paper(s) to Extract

In [ ]:
def show_extraction_queue(limit=20):
    """Show papers that need extraction."""
    status = ExtractionService.get_extraction_status()
    print("=" * 60)
    print("EXTRACTION STATUS")
    print("=" * 60)
    print(f"Total papers: {status.total_papers}")
    print(f"With full text: {status.papers_with_full_text}")
    print(f"With abstract: {status.papers_with_abstract}")
    print(f"Already extracted: {status.papers_with_extraction}")
    print(f"Needing extraction: {status.papers_needing_extraction}")
    print(f"Coverage: {status.extraction_coverage_percent:.1f}%")
    print()
    
    papers = ExtractionService.get_papers_needing_extraction(limit=limit)
    print(f"Next {len(papers)} papers to extract:")
    print("-" * 60)
    for i, p in enumerate(papers, 1):
        title = p['title'][:50] + '...' if len(p['title']) > 50 else p['title']
        text_type = "[FULL TEXT]" if p['has_full_text'] else "[ABSTRACT]"
        print(f"{i:2}. ID {p['id']:3}: {title} {text_type}")
    
    return papers

queue = show_extraction_queue()

In [ ]:
# Select a paper to extract
# Change this to extract a specific paper
PAPER_ID = queue[0]['id'] if queue else 1

# Load paper details
paper = PaperService.get(PAPER_ID)
if paper:
    print(f"Selected paper ID: {PAPER_ID}")
    print(f"Title: {paper['title']}")
    print(f"Authors: {paper.get('authors', 'N/A')}")
    print(f"Year: {paper.get('year', 'N/A')}")
    print(f"Has abstract: {'Yes' if paper.get('abstract') else 'No'}")
    print(f"Has full text: {'Yes' if paper.get('full_text') else 'No'}")
else:
    print(f"Paper {PAPER_ID} not found!")

## Cell 4: Single Paper Extraction (Using Service)

Uses the improved prompt that extracts:
- Paper type, topics, summary
- Key findings with quantitative data
- Detailed methodology
- **Discussion summary** (authors' interpretations)
- **Research context** (problem, novelty, limitations, significance)
- **Future directions**

In [ ]:
async def extract_single_paper(paper_id: int, force: bool = False):
    """Extract a paper using the ExtractionService with verbose output."""
    print("=" * 70)
    print(f"EXTRACTING PAPER {paper_id}")
    print("=" * 70)
    
    # Run extraction with verbose mode
    result = await ExtractionService.extract_paper(
        paper_id=paper_id,
        backend='ollama',
        force=force,
        verbose=True
    )
    
    print(f"\n{'='*70}")
    print("EXTRACTION RESULT")
    print("="*70)
    print(f"Success: {result.success}")
    print(f"Model: {result.extractor_model}")
    print(f"Time: {result.elapsed_seconds:.1f}s" if result.elapsed_seconds else "Time: N/A")
    
    if result.success:
        print(f"\n📋 PAPER TYPE: {result.paper_type}")
        
        print(f"\n🏷️ TOPICS:")
        for t in (result.topics or []):
            print(f"   • {t}")
        
        print(f"\n📝 ONE-SENTENCE SUMMARY:")
        print(f"   {result.one_sentence_summary}")
        
        print(f"\n🔬 KEY FINDINGS ({len(result.key_findings or [])}):") 
        for i, f in enumerate(result.key_findings or [], 1):
            print(f"   {i}. {f}")
        
        print(f"\n🧪 METHODOLOGY:")
        print(f"   {result.methodology_summary}")
        
        print(f"\n💬 DISCUSSION SUMMARY:")
        print(f"   {result.discussion_summary}")
        
        if result.research_context:
            print(f"\n🎯 RESEARCH CONTEXT:")
            ctx = result.research_context
            print(f"   Problem: {ctx.get('problem_addressed', 'N/A')}")
            print(f"   Novelty: {ctx.get('novelty', 'N/A')}")
            print(f"   Limitations: {ctx.get('limitations', 'N/A')}")
            print(f"   Significance: {ctx.get('significance', 'N/A')}")
        
        print(f"\n🔮 FUTURE DIRECTIONS:")
        for d in (result.future_directions or []):
            print(f"   • {d}")
    else:
        print(f"\n❌ Error: {result.error}")
        if result.raw_response:
            print(f"\n📤 Raw LLM response (first 1000 chars):")
            print(result.raw_response[:1000])
    
    return result

# Run extraction (set force=True to re-extract)
extraction = await extract_single_paper(PAPER_ID, force=True)

## Cell 5: Review & Store Extraction

In [ ]:
# The extraction is automatically stored if successful
# This cell shows what was stored

if extraction and extraction.success:
    print("✓ Extraction was automatically stored in the database")
    print(f"\nPaper ID: {extraction.paper_id}")
    print(f"Model used: {extraction.extractor_model}")
else:
    print("❌ Extraction failed - nothing stored")

## Cell 6: Batch Extraction with Progress

In [ ]:
async def batch_extract(paper_ids: list, delay: float = 2.0):
    """Extract multiple papers with progress bar."""
    results = {'success': [], 'failed': [], 'skipped': []}
    
    print(f"Extracting {len(paper_ids)} papers...")
    print(f"Delay between extractions: {delay}s")
    print(f"Estimated time: {len(paper_ids) * 50 / 60:.1f} minutes")
    print()
    
    for paper_id in tqdm(paper_ids, desc="Extracting"):
        try:
            result = await ExtractionService.extract_paper(
                paper_id=paper_id,
                backend='ollama',
                force=False  # Skip already extracted
            )
            
            if result.success:
                if "Already extracted" in (result.error or ""):
                    results['skipped'].append(paper_id)
                else:
                    results['success'].append(paper_id)
            else:
                results['failed'].append({'id': paper_id, 'error': result.error})
            
            if delay > 0:
                await asyncio.sleep(delay)
                
        except Exception as e:
            results['failed'].append({'id': paper_id, 'error': str(e)})
    
    print()
    print("=" * 60)
    print("BATCH EXTRACTION COMPLETE")
    print("=" * 60)
    print(f"✓ Success: {len(results['success'])}")
    print(f"⊘ Skipped (already extracted): {len(results['skipped'])}")
    print(f"✗ Failed: {len(results['failed'])}")
    
    if results['failed']:
        print(f"\nFailed papers:")
        for f in results['failed']:
            print(f"  Paper {f['id']}: {f['error']}")
    
    return results

# Select papers to extract
papers_to_extract = [p['id'] for p in queue[:5]]  # First 5 from queue
print(f"Will extract papers: {papers_to_extract}")
print("Run next cell to start batch extraction")

In [ ]:
# Run batch extraction
batch_results = await batch_extract(papers_to_extract, delay=2.0)

## Cell 7: Review Extraction Results

In [ ]:
def show_full_extraction(paper_id: int):
    """Show full extraction details for a paper including extended fields."""
    with get_session() as session:
        content = session.query(PaperContent).filter(
            PaperContent.paper_id == paper_id
        ).first()
        
        if not content:
            print(f"No extraction found for paper {paper_id}")
            return
        
        paper = session.query(Paper).filter(Paper.id == paper_id).first()
        
        print("=" * 70)
        print(f"EXTRACTION FOR PAPER {paper_id}")
        print("=" * 70)
        print(f"Title: {paper.title if paper else 'Unknown'}")
        print(f"\n📋 Paper Type: {content.paper_type}")
        
        print(f"\n🏷️ Topics:")
        for t in (content.topics or []):
            print(f"   • {t}")
        
        print(f"\n📝 One-Sentence Summary:")
        print(f"   {content.one_sentence_summary}")
        
        print(f"\n🔬 Key Findings:")
        for i, f in enumerate(content.key_findings or [], 1):
            print(f"   {i}. {f}")
        
        print(f"\n🧪 Methodology:")
        print(f"   {content.methodology_summary}")
        
        # Extended fields from structured_data
        if content.structured_data:
            sd = content.structured_data
            
            if sd.get('discussion_summary'):
                print(f"\n💬 Discussion Summary:")
                print(f"   {sd['discussion_summary']}")
            
            if sd.get('research_context'):
                print(f"\n🎯 Research Context:")
                ctx = sd['research_context']
                print(f"   Problem: {ctx.get('problem_addressed', 'N/A')}")
                print(f"   Novelty: {ctx.get('novelty', 'N/A')}")
                print(f"   Limitations: {ctx.get('limitations', 'N/A')}")
                print(f"   Significance: {ctx.get('significance', 'N/A')}")
            
            if sd.get('future_directions'):
                print(f"\n🔮 Future Directions:")
                for d in sd['future_directions']:
                    print(f"   • {d}")
        
        print(f"\n📅 Extracted: {content.extraction_date}")
        print(f"🤖 Model: {content.extractor_model}")

# View the paper we just extracted
show_full_extraction(PAPER_ID)

## Cell 8: Text Quality Assessment

Checks for papers with text quality issues (ILL contamination, missing abstracts) that may need vision-based extraction.

In [ ]:
def show_quality_issues():
    """Show papers with text quality issues."""
    print("=" * 60)
    print("TEXT QUALITY ASSESSMENT")
    print("=" * 60)
    
    issues = ExtractionService.get_papers_with_quality_issues(limit=50)
    
    contaminated = [p for p in issues if p['quality'] == 'contaminated']
    suspect = [p for p in issues if p['quality'] == 'suspect']
    
    print(f"\n🔴 CONTAMINATED (ILL/library metadata): {len(contaminated)}")
    for p in contaminated:
        print(f"   ID {p['id']}: {p['title'][:45]}...")
        print(f"      Recommendation: {p['recommendation']}")
    
    print(f"\n🟡 SUSPECT (may need attention): {len(suspect)}")
    for p in suspect:
        print(f"   ID {p['id']}: {p['title'][:45]}...")
        print(f"      Issues: {p['issues']}")
    
    print(f"\n📊 Summary:")
    print(f"   Need vision extraction: {len([p for p in issues if p['recommendation'] == 'vision'])}")
    print(f"   Need enrichment: {len([p for p in issues if p['recommendation'] == 'enrich'])}")
    
    return issues

quality_issues = show_quality_issues()

## Cell 9: Vision Extraction (for Contaminated PDFs)

For papers with ILL contamination or important figures, convert PDF to images and use Claude Code's vision capability.

**No extra API cost** - uses your existing Claude Code session.

In [ ]:
from pdf2image import convert_from_path

def prepare_for_vision(paper_id: int, pages: tuple = (1, 5), dpi: int = 150):
    """Convert PDF pages to images for Claude Code vision extraction."""
    print("=" * 60)
    print(f"PREPARING PAPER {paper_id} FOR VISION EXTRACTION")
    print("=" * 60)
    
    paper = PaperService.get(paper_id)
    if not paper:
        print(f"✗ Paper {paper_id} not found")
        return None
    
    pdf_path = paper.get('file_path')
    if not pdf_path or not Path(pdf_path).exists():
        print(f"✗ PDF not found: {pdf_path}")
        return None
    
    print(f"Title: {paper['title']}")
    print(f"PDF: {pdf_path}")
    
    output_dir = Path(f'/tmp/paper_{paper_id}_pages')
    output_dir.mkdir(exist_ok=True)
    
    for f in output_dir.glob('*.png'):
        f.unlink()
    
    print(f"\nConverting pages {pages[0]}-{pages[1]} at {dpi} DPI...")
    try:
        images = convert_from_path(pdf_path, dpi=dpi, first_page=pages[0], last_page=pages[1])
        
        for i, img in enumerate(images):
            out_path = output_dir / f'page_{pages[0] + i}.png'
            img.save(out_path, 'PNG')
            print(f"  ✓ {out_path.name} ({img.width}x{img.height})")
        
        print(f"\n✓ {len(images)} pages saved to {output_dir}")
        print(f"\n📋 Next steps:")
        print(f"   1. In Claude Code, ask: 'Read the images in {output_dir}'")
        print(f"   2. Claude will extract information visually")
        print(f"   3. Store via MCP: mcp__literature__store_extraction(...)")
        
        return output_dir
        
    except Exception as e:
        print(f"✗ Conversion failed: {e}")
        return None

# Select a paper that needs vision extraction
VISION_PAPER_ID = quality_issues[0]['id'] if quality_issues else 175
output_path = prepare_for_vision(VISION_PAPER_ID, pages=(1, 5))